In [1]:
!pip install -q -U transformers peft accelerate safetensors bitsandbytes>=0.46.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [2]:
import os
import json
import shutil
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)
from peft import PeftModel

# Change this ONLY if your Kaggle dataset has a different path
ADAPTER_DIR = "/kaggle/input/datasets/pernavjain/python-bugs"

# Final merged model will be stored here
OUTPUT_DIR = "/kaggle/working/qwen25-coder-3b-python-bugs"

print("Adapter directory:", ADAPTER_DIR)
print("Output directory :", OUTPUT_DIR)
print("CUDA available   :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Adapter directory: /kaggle/input/datasets/pernavjain/python-bugs
Output directory : /kaggle/working/qwen25-coder-3b-python-bugs
CUDA available   : True
GPU: Tesla T4


In [3]:
print("Files in adapter directory:\n")

for f in sorted(os.listdir(ADAPTER_DIR)):
    path = os.path.join(ADAPTER_DIR, f)

    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024 ** 2)
        print(f"{f:40s} {size_mb:10.2f} MB")

Files in adapter directory:

README.md                                      0.00 MB
adapter_config.json                            0.00 MB
adapter_model.safetensors                     28.16 MB
chat_template.jinja                            0.00 MB
tokenizer.json                                10.89 MB
tokenizer_config.json                          0.00 MB


In [4]:
adapter_config_path = os.path.join(
    ADAPTER_DIR,
    "adapter_config.json"
)

with open(adapter_config_path, "r") as f:
    adapter_config = json.load(f)

BASE_MODEL = adapter_config["base_model_name_or_path"]

print("Base model:")
print(BASE_MODEL)

Base model:
unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit


In [5]:
print("LoRA configuration:\n")

for key, value in adapter_config.items():
    print(f"{key}: {value}")

LoRA configuration:

alora_invocation_tokens: None
alpha_pattern: {}
arrow_config: None
auto_mapping: {'base_model_class': 'Qwen2ForCausalLM', 'parent_library': 'transformers.models.qwen2.modeling_qwen2', 'unsloth_fixed': True}
base_model_name_or_path: unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit
bias: none
corda_config: None
ensure_weight_tying: False
eva_config: None
exclude_modules: None
fan_in_fan_out: False
inference_mode: True
init_lora_weights: True
kasa_config: None
layer_replication: None
layers_pattern: None
layers_to_transform: None
loftq_config: {}
lora_alpha: 32
lora_bias: False
lora_dropout: 0.0
lora_ga_config: None
megatron_config: None
megatron_core: megatron.core
modules_to_save: None
monteclora_config: None
peft_type: LORA
peft_version: 0.21.0
qalora_group_size: 16
r: 16
rank_pattern: {}
revision: None
target_modules: ['q_proj', 'k_proj', 'o_proj', 'v_proj']
target_parameters: None
task_type: CAUSAL_LM
trainable_token_indices: None
use_bdlora: None
use_dora: False
use_q

In [6]:
tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_DIR,
    trust_remote_code=True,
)

print("Tokenizer loaded successfully")
print("Vocab size:", len(tokenizer))
print("EOS token:", tokenizer.eos_token)
print("PAD token:", tokenizer.pad_token)

Tokenizer loaded successfully
Vocab size: 151666
EOS token: <|im_end|>
PAD token: <|PAD_TOKEN|>


In [7]:
dtype = torch.float16

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
)

print("Base model loaded successfully")

config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.05GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

Base model loaded successfully


In [8]:
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
)

print("LoRA adapter loaded successfully")

LoRA adapter loaded successfully


In [9]:
print(model)
print("Trainable parameters:")
model.print_trainable_parameters()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048, padding_idx=151665)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [10]:
print("Merging LoRA adapter into base model...")

model = model.merge_and_unload()

print("✅ LoRA merge completed")
print("Model class:", model.__class__.__name__)

Merging LoRA adapter into base model...


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:377: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


✅ LoRA merge completed
Model class: Qwen2ForCausalLM


In [11]:
if os.path.exists(OUTPUT_DIR):
    print("Removing previous output directory...")
    shutil.rmtree(OUTPUT_DIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Output directory ready:", OUTPUT_DIR)

Output directory ready: /kaggle/working/qwen25-coder-3b-python-bugs


In [12]:
print("Saving merged model...")

model.save_pretrained(
    OUTPUT_DIR,
    safe_serialization=True,
    save_original_format=False,
    max_shard_size="2GB",
)

print("✅ Model weights saved")

Saving merged model...


Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Model weights saved


In [13]:
tokenizer.save_pretrained(OUTPUT_DIR)

print("✅ Tokenizer saved")

✅ Tokenizer saved


In [14]:
print("\nFinal model files:\n")

total_size = 0

for root, dirs, files in os.walk(OUTPUT_DIR):
    for file in sorted(files):
        path = os.path.join(root, file)
        size_mb = os.path.getsize(path) / (1024 ** 2)
        total_size += size_mb

        print(f"{file:50s} {size_mb:10.2f} MB")

print("\nTotal size: {:.2f} MB ({:.2f} GB)".format(
    total_size,
    total_size / 1024
))


Final model files:

chat_template.jinja                                      0.00 MB
config.json                                              0.00 MB
generation_config.json                                   0.00 MB
model-00001-of-00002.safetensors                      1899.30 MB
model-00002-of-00002.safetensors                        60.14 MB
model.safetensors.index.json                             0.15 MB
tokenizer.json                                          10.89 MB
tokenizer_config.json                                    0.00 MB

Total size: 1970.49 MB (1.92 GB)


In [15]:
del model
del base_model

import gc
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Original model objects cleared from memory.")

Original model objects cleared from memory.


In [16]:
test_tokenizer = AutoTokenizer.from_pretrained(
    OUTPUT_DIR,
    trust_remote_code=True,
)

test_model = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

print("✅ Standalone merged model loaded successfully")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

✅ Standalone merged model loaded successfully


In [17]:
# Cell 17 — Test generation

messages = [
    {
        "role": "user",
        "content": "Write a Python function that checks whether a number is prime."
    }
]

# Create the chat-formatted prompt as text first
prompt = test_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print("Prompt:")
print(prompt)

# Tokenize the formatted prompt
inputs = test_tokenizer(
    prompt,
    return_tensors="pt",
)

# Move tensors to the same device as the model
inputs = {
    k: v.to(test_model.device)
    for k, v in inputs.items()
}

print("\nInput shape:", inputs["input_ids"].shape)

# Generate
with torch.no_grad():
    outputs = test_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
    )

# Remove the original prompt tokens
generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

# Decode only the generated answer
answer = test_tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True,
)

print("\n" + "=" * 60)
print("MODEL OUTPUT")
print("=" * 60)
print(answer)

Prompt:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Write a Python function that checks whether a number is prime.<|im_end|>
<|im_start|>assistant


Input shape: torch.Size([1, 41])


[transformers] Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



MODEL OUTPUT
Here's a Python function to check if a number is prime:

```python
def is_prime(n):
    # Check if the number is less than 2, which are not prime numbers
    if n < 2:
        return False
    
    # Check for factors from 2 to the square root of n
    for i in range(2, int(n**0.5) + 1):
        # If n is divisible by any number in this range, it's not prime
        if n % i == 0:
            return False
    
    # If no factors were found, n is prime
    return True
```

### Explanation:

1. **Initial Check**: The function first checks if `n`


In [18]:
# Cell 18 — Verify the saved model is truly standalone

print("Model class:", test_model.__class__.__name__)

is_peft = hasattr(test_model, "peft_config")

print("Is PEFT model:", is_peft)

if not is_peft:
    print("✅ This is a standalone merged model.")
else:
    print("⚠️ Model still contains PEFT configuration.")

Model class: Qwen2ForCausalLM
Is PEFT model: False
✅ This is a standalone merged model.


In [19]:
# Cell 19 — Inspect final model directory

import os

total_size = 0

print("Final model files:\n")

for file in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, file)

    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024 ** 2)
        total_size += size_mb
        print(f"{file:50s} {size_mb:10.2f} MB")

print("\n" + "=" * 60)
print(f"Total size: {total_size:.2f} MB")
print(f"Total size: {total_size / 1024:.2f} GB")

Final model files:

chat_template.jinja                                      0.00 MB
config.json                                              0.00 MB
generation_config.json                                   0.00 MB
model-00001-of-00002.safetensors                      1899.30 MB
model-00002-of-00002.safetensors                        60.14 MB
model.safetensors.index.json                             0.15 MB
tokenizer.json                                          10.89 MB
tokenizer_config.json                                    0.00 MB

Total size: 1970.49 MB
Total size: 1.92 GB


In [20]:
# Cell 20 — Final standalone loading test

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

del test_model
del test_tokenizer

import gc
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Loading only from merged model directory...")

final_tokenizer = AutoTokenizer.from_pretrained(
    OUTPUT_DIR,
    trust_remote_code=True,
)

final_model = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

print("\n✅ FINAL CHECK PASSED")
print("Model:", final_model.__class__.__name__)
print("Location:", OUTPUT_DIR)

Loading only from merged model directory...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


✅ FINAL CHECK PASSED
Model: Qwen2ForCausalLM
Location: /kaggle/working/qwen25-coder-3b-python-bugs


In [21]:
# Cell 21 — Final generation test

messages = [
    {
        "role": "user",
        "content": "Find and explain the bug in this Python code:\n\n"
                   "def add_numbers(a, b):\n"
                   "    return a - b"
    }
]

prompt = final_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = final_tokenizer(
    prompt,
    return_tensors="pt",
)

inputs = {
    k: v.to(final_model.device)
    for k, v in inputs.items()
}

with torch.no_grad():
    outputs = final_model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

print(final_tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True,
))

[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The bug in the provided Python function `add_numbers` is that it performs subtraction instead of addition. The function is defined to take two arguments, `a` and `b`, and returns their difference (`a - b`). To fix this, you should change the operation from subtraction to addition.

Here's the corrected version of the function:

```python
def add_numbers(a, b):
    return a + b
```

This will correctly compute the sum of `a` and `b`.


In [23]:
import kagglehub
from pathlib import Path

MODEL_DIR = "/kaggle/working/qwen25-coder-3b-python-bugs"

# Kaggle model handle:
# username/model-slug/framework/variation-slug
HANDLE = "pernavjain/bug-whisper-qwen25-coder-3b/pyTorch/4bit-bnb"

# Basic check
model_dir = Path(MODEL_DIR)

if not model_dir.exists():
    raise FileNotFoundError(f"Model directory not found: {MODEL_DIR}")

print("Uploading:")
print(MODEL_DIR)
print("\nTo:")
print(HANDLE)
print()

# Upload model
result = kagglehub.model_upload(
    HANDLE,
    MODEL_DIR,
    version_notes="Initial standalone merged 4-bit Bug Whisper model",
    license_name="Apache 2.0",
)

print("\n✅ Upload complete")
print(result)

Uploading:
/kaggle/working/qwen25-coder-3b-python-bugs

To:
pernavjain/bug-whisper-qwen25-coder-3b/pyTorch/4bit-bnb

Uploading Model https://api.kaggle.com/models/pernavjain/bug-whisper-qwen25-coder-3b/pyTorch/4bit-bnb ...
Model 'bug-whisper-qwen25-coder-3b' does not exist or access is forbidden for user 'pernavjain'. Creating or handling Model...
Starting upload for file /kaggle/working/qwen25-coder-3b-python-bugs/generation_config.json


Uploading: 100%|██████████| 266/266 [00:00<00:00, 415B/s]

Upload successful: /kaggle/working/qwen25-coder-3b-python-bugs/generation_config.json (266B)
Starting upload for file /kaggle/working/qwen25-coder-3b-python-bugs/model.safetensors.index.json



Uploading: 100%|██████████| 158k/158k [00:00<00:00, 238kB/s]

Upload successful: /kaggle/working/qwen25-coder-3b-python-bugs/model.safetensors.index.json (154KB)
Starting upload for file /kaggle/working/qwen25-coder-3b-python-bugs/model-00001-of-00002.safetensors


...


Uploading: 100%|██████████| 63.1M/63.1M [00:01<00:00, 47.8MB/s]

Upload successful: /kaggle/working/qwen25-coder-3b-python-bugs/model-00002-of-00002.safetensors (60MB)
Starting upload for file /kaggle/working/qwen25-coder-3b-python-bugs/chat_template.jinja



Uploading: 100%|██████████| 2.51k/2.51k [00:00<00:00, 3.93kB/s]

Upload successful: /kaggle/working/qwen25-coder-3b-python-bugs/chat_template.jinja (2KB)


Your model instance has been created.
Files are being processed...
See at: https://api.kaggle.com/models/pernavjain/bug-whisper-qwen25-coder-3b/pyTorch/4bit-bnb

✅ Upload complete
None
